# 113장 · 파일 백업과 세션 데이터 복원

백업 파일의 존재만으로는 복구 여부를 알 수 없습니다. 다른 위치에서 사본을 열고 필요한 레코드가 그대로 읽히는지 확인합니다.

임시 세션 사본을 별도 위치에 복원하고 저장된 요청을 다시 조회합니다. 원본 보존, 사본의 바이트 일치, 실제 업무 데이터의 복구 범위를 구분해 기록합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 팀별 변형을 시도한 뒤 해당 셀 아래에서 예시를 펼쳐 비교할 수 있습니다.


## 배포 명세에 복구할 마지막 요청 적기

112장에서 지정한 볼륨에 어떤 요청이 남아야 하는지 적습니다. 요청 하나가 빠진 사본을 만들어 단순히 파일을 읽을 수 있는 경우와 최신 요청까지 복구한 경우를 구별합니다.

이 프로젝트의 노트북은 한 폴더에서 열어 주세요. 첫 셀은 `team-project/final/chapter112.json`을 먼저 찾습니다. 팀 파일이 없으면 [이 장의 시작 예시](chapter113-start.json)를 노트북 옆에 내려받으세요. 출력 첫 줄에서 어느 파일을 읽었는지 확인할 수 있습니다.

In [ ]:
from pathlib import Path
import json
import re

here = Path.cwd()
project_root = here.parent if re.fullmatch(r"chapter[0-9]+", here.name) else here
team_file = project_root / "team-project/final/chapter112.json"
sample_file = here / "chapter113-start.json"
input_file = team_file if team_file.is_file() else sample_file
if not input_file.is_file():
    raise FileNotFoundError("이 장의 시작 예시를 내려받아 노트북 옆에 두세요.")
incoming = json.loads(input_file.read_text(encoding="utf-8"))
print("팀 기록" if input_file == team_file else "교재 예시", input_file.name)
print(incoming['deployment'])

교재 예시 chapter113-start.json
{'local_candidate_ok': True, 'port': '127.0.0.1:8104:8104', 'mount': 'p1-json:/app/input106/project-ui/backend/runtime-data', 'digest': None, 'status': '미배포 예시'}


두 번째 줄은 읽은 파일의 `deployment` 값입니다.

## 별도 경로로 세션 사본 복원

복구 기록 예시　① 가상 세션 파일을 준비 ② 별도 디렉터리에 복구 ③ 복구 직후 원본과 바이트 대조 ④ 새 위치에서 마지막 요청 ID 읽기. 운영 상황에서는 복구 시점과 새 쓰기를 멈췄는지 먼저 확인해야 합니다. 아래 셀은 자신이 만든 임시 폴더 안에서만 파일을 만듭니다.

**원본과 별도 위치에 복원한 사본**　왼쪽 보관함에 원본이 남아 있는 동안 오른쪽 사본을 열어 봅니다. 복구 검증은 사본을 실제로 읽고 필요한 값을 대조하는 작업입니다.

### 셀 01 · 세션 사본의 복원과 재조회

읽기 확인은 복구 대상에서 수행합니다. 원본 내용으로 답을 출력하면 잘못된 복구 경로도 성공으로 보입니다.


In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import json
from shutil import copyfile

with TemporaryDirectory() as temp:
    root = Path(temp)
    source = root / "source.json"
    source.write_text(json.dumps({"owner": "alice", "requests": ["r5", "r6"]}), encoding="utf-8")
    target_dir = root / "restored"
    target_dir.mkdir()
    target = target_dir / "session.json"
    copyfile(source, target)
    restored = json.loads(target.read_text(encoding="utf-8"))
    print(target.read_bytes() == source.read_bytes(), restored["requests"][-1])

True r6


True와 r6가 나옵니다. 첫 값은 현재 두 사본의 바이트 일치이고 둘째 값은 복구한 사본을 읽은 결과입니다. 셀이 끝나면 이 셀의 임시 디렉터리는 정리됩니다.

복원한 사본에서 요청 r6을 다시 읽은 것이 이번 검사의 범위입니다. 운영 복구 시간을 제시하려면 운영 백업을 대상으로 복원을 수행하고, 필요한 요청을 다시 조회할 때까지 걸린 시간을 기록해야 합니다. 로컬 사본의 성공과 운영 훈련 결과를 분리합니다.


### True와 r6이 확인한 복원 결과의 차이

True는 복사한 두 파일을 바이트 단위로 비교한 결과입니다. r6은 복구한 위치에서 JSON을 다시 읽고 마지막 요청 ID를 꺼낸 값입니다. 바이트가 같다는 검사와 복구 파일을 실제로 파싱해 원하는 기록을 찾는 검사를 함께 수행한 셈입니다.


## 복원한 레코드의 내용과 중복 검사

제공 가상 복구 기록에서는 alice가 유지한 요청이 r2~r6였고 다른 사용자의 같은 이름 세션은 0턴이었습니다. 복구 뒤에도 만료 시각은 그대로 두었습니다. 최근 r6을 다시 쓰려는 요청은 거부했고, 보존 범위를 벗어난 r0까지 영구 차단하지는 않았습니다. “파일이 돌아왔다”와 “오래된 중복 요청이 모두 차단된다”는 각각 다른 주장입니다.


### 팀 참고 예시 · 복원 범위 기록


In [2]:
record = {'original_unchanged_after_replay': True,
 'actual_restore_of_operational_data': False}
print("가상 원본 보존", record["original_unchanged_after_replay"])
print("운영 자료 실제 복구", record["actual_restore_of_operational_data"])

가상 원본 보존 True
운영 자료 실제 복구 False


가상 원본 보존은 True, 운영 자료 복구는 False입니다. 팀의 실환경 기록에는 복구 대상과 확인 시간을 다시 적습니다.


#### 복원 성공을 기록할 때

<mark>가상 사본의 레코드가 복원된 결과와 운영 자료의 복원 여부를 따로 남깁니다.</mark> 건수만 맞아도 잘못된 세션을 되살렸을 수 있습니다. 원본을 보존한 위치와 복원본의 레코드 ID·내용을 함께 기록하세요.

선수 개념 · 75장 SQLite 백업과 복구 검증

## 파일은 읽히지만 마지막 요청이 빠진 복구본 찾아내기

백업 파일이 JSON으로 열렸다고 해서 r6까지 복구된 것은 아닙니다. 원본은 보존하고, 오래된 사본을 실수로 고른 경우를 한 번 만들어 복구 인수 기준을 세웁니다.

### 셀 04 · 두 복구 후보의 요청 ID 대조

별도의 임시 폴더에 최근 기록 사본과 마지막 요청이 빠진 사본을 각각 둡니다. 마지막 요청 하나가 어느 파일에서 빠지는지 비교합니다.

In [3]:
original_ids = ["r2", "r3", "r4", "r5", "r6"]
with TemporaryDirectory() as folder:
    root = Path(folder)
    latest = root / "latest.json"
    older = root / "older.json"
    latest.write_text(json.dumps(original_ids), encoding="utf-8")
    older.write_text(json.dumps(original_ids[:-1]), encoding="utf-8")
    reopened = json.loads(older.read_text(encoding="utf-8"))
    expected_ids = json.loads(latest.read_text(encoding="utf-8"))
    missing_ids = [request_id for request_id in expected_ids if request_id not in reopened]
    print("사본의 마지막 요청", reopened[-1])
    print("누락된 요청", missing_ids)

사본의 마지막 요청 r5
누락된 요청 ['r6']


오래된 사본도 정상 JSON이고 네 건을 읽지만 r6이 없습니다. 이 사본을 사용하면 r6부터 다시 처리해야 하는지 먼저 조사해야 합니다.

### 팀 복구 기록에 덧붙일 내용

같은 검사에서 소유자와 세션, 보존 기간, 중복 요청 처리도 각각 확인하세요. 복구본의 행 개수만 적으면 다른 사용자의 다섯 턴을 가져온 경우를 가려내지 못합니다. 새 쓰기 허용은 팀이 필요한 범위의 기록을 대조한 뒤 결정합니다.

<mark>복구 성공을 말할 때는 어느 시점의 어떤 데이터까지 읽혔는지 적습니다.</mark> 이 실험의 오래된 사본으로 확인한 범위는 r5까지입니다. 실제 데이터 손실이나 실서비스 복구 시간을 측정한 값은 없습니다.

114장에는 원하는 요청 ID를 찾지 못한 인수자가 어디까지 확인했는지 기록하는 예를 실습니다.

## 팀 사례에 적용하기 · 배포 명세에 복구할 마지막 요청 적기

위 예제에 첫 셀에서 읽어 둔 팀 기록을 넣어 봅니다.

In [ ]:
expected_ids = [incoming["lookup"]["request"]]
with TemporaryDirectory() as folder:
    backup = Path(folder) / "previous.json"
    backup.write_text(json.dumps(expected_ids[:-1]), encoding="utf-8")
    restored = json.loads(backup.read_text(encoding="utf-8"))
missing = [key for key in expected_ids if key not in restored]
new = {**incoming, "restore": {"mount": incoming["deployment"]["mount"],
       "restore_missing": missing, "environment": "임시 JSON"}}
print(missing, new["restore"]["environment"])

['team-99'] 임시 JSON


restore_missing에 팀 요청 ID가 남으면 그 사본은 최신 복구 지점을 충족하지 못합니다. 실습의 JSON 사본과 운영 볼륨의 실제 복구 결과를 따로 남깁니다.

In [ ]:
output_file = project_root / "team-project/final/chapter113.json"
output_file.parent.mkdir(parents=True, exist_ok=True)
with output_file.open("a+", encoding="utf-8") as handle:
    handle.seek(0)
    previous_text = handle.read()
    if previous_text and json.loads(previous_text) != new:
        raise FileExistsError("같은 장의 다른 결과가 있습니다. 기존 JSON을 다른 이름으로 보관한 뒤 다시 실행하세요.")
    if not previous_text:
        json.dump(new, handle, ensure_ascii=False, indent=2)
        handle.write("\n")
print(output_file.relative_to(project_root).as_posix())

team-project/final/chapter113.json


114장 첫 셀이 이 파일을 읽습니다. 입력을 바꿔 다시 실행할 때는 기존 결과 파일의 이름을 바꿔 보관하세요.

## 돌아보기

114장에 인계할 기록은 복구 대상·복구한 위치·읽어 확인한 값·아직 모르는 범위입니다. 운영 장비의 재시작 시각과 실제 사용자 영향은 이 로컬 연습만으로 채우지 않습니다.
